# Extracción de tenis deportivos para mujer de Calzatodo
### Evidencia de aprendizaje 2 — Programación para el Análisis de Datos

**Integrantes:** Estefanía Martínez Guzmán · Carlos Andrés Osorio Agudelo · Valentina Pérez Flórez

Este notebook retoma la extracción de la Evidencia de aprendizaje 1 y añade el
registro del flujo de trabajo con Git y GitHub. Se puede ejecutar de principio a
fin en Google Colab o en cualquier entorno con Python 3.11 o superior.

## Objetivo
Obtener y organizar nombres, marcas, referencia y precios de las publicaciones de
las dos primeras páginas de la categoría de tenis deportivos para mujer, y
versionar el código y los resultados en un repositorio de GitHub.


## 1. Herramientas

`Requests` descarga el HTML de las páginas, `BeautifulSoup` localiza los bloques
JSON-LD de tipo `ItemList` y `json` los interpreta. `Pandas` organiza los
registros en una tabla y permite validar la calidad de los datos.


In [1]:
import json
import os
import subprocess
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

try:
    from google.colab import userdata

    EN_COLAB = True
except ImportError:
    EN_COLAB = False

URL_CATEGORIA = "https://www.calzatodo.com.co/es/mujeres/tenis/deportivo"
ARCHIVO_CSV = "calzatodo_tenis_deportivos_mujer.csv"
REPO_URL = "https://github.com/Estefa-M/EA2_Web_Scraping.git"
REPO_DIR = "EA2_Web_Scraping"

print("Entorno Colab:", EN_COLAB)
print("Pandas:", pd.__version__)


Entorno Colab: False
Pandas: 2.3.3


## 2. Extracción

Se consultan las páginas 1 y 2 de la categoría. Cada página declara un bloque
JSON-LD `ItemList` con veinte elementos; como la página 1 repite publicaciones de
la portada, el total se reduce al eliminar las URL duplicadas.

En la primera ejecución de la Evidencia de aprendizaje 1 se solicitaba la página 1
sin el parámetro `page`, porque era la URL base la que entregaba el bloque. El
sitio cambió ese comportamiento: hoy la URL base ya no incluye el `ItemList` y la
página 1 debe pedirse con `page=1`. El código se ajustó a ese cambio, que es la
mejora aplicada al proyecto.


In [2]:
productos = []
paginas = [1, 2]
errores_http = []

for numero, pagina in enumerate(paginas, start=1):
    response = requests.get(URL_CATEGORIA, params={"page": pagina}, timeout=30)
    if response.status_code != 200:
        errores_http.append(response.status_code)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")
    encontrados = 0

    for bloque in soup.find_all("script", type="application/ld+json"):
        try:
            datos = json.loads(bloque.get_text())
        except json.JSONDecodeError:
            # Si un bloque no es JSON valido se omite y se continua con el resto.
            continue

        if isinstance(datos, dict) and datos.get("@type") == "ItemList":
            for elemento in datos.get("itemListElement", []):
                producto = elemento.get("item", {})
                ofertas = producto.get("offers") or {}

                productos.append({
                    "Nombre": producto.get("name"),
                    "Marca": (producto.get("brand") or {}).get("name"),
                    "SKU de referencia": producto.get("sku"),
                    "Precio mínimo": ofertas.get("lowPrice"),
                    "Precio máximo": ofertas.get("highPrice"),
                    "Moneda": ofertas.get("priceCurrency"),
                    "URL": producto.get("@id"),
                })
                encontrados += 1

    print(f"Página {numero} procesada: {encontrados} elementos leídos")
    if numero < len(paginas):
        time.sleep(2)

print(f"\nRegistros descargados: {len(productos)}")
print(f"Respuestas HTTP distintas de 200: {len(errores_http)}")


Página 1 procesada: 20 elementos leídos


Página 2 procesada: 20 elementos leídos

Registros descargados: 40
Respuestas HTTP distintas de 200: 0


## 3. Preparación y comprobación de los datos

Se construye la tabla, se eliminan las publicaciones con URL repetida y se
convierten los precios a valores numéricos. Después se calculan las métricas de
éxito definidas para el proyecto.


In [3]:
df_productos = pd.DataFrame(productos)

if df_productos.empty:
    raise ValueError("No se encontraron productos. Revise la extracción.")

df_productos = df_productos.drop_duplicates(subset=["URL"]).reset_index(drop=True)

for columna in ["Precio mínimo", "Precio máximo"]:
    df_productos[columna] = pd.to_numeric(df_productos[columna], errors="coerce")

columnas = ["Nombre", "Marca", "SKU de referencia", "Precio mínimo",
            "Precio máximo", "Moneda", "URL"]
celdas_totales = len(df_productos) * len(columnas)
celdas_completas = int(df_productos[columnas].notna().sum().sum())
completitud = celdas_completas / celdas_totales * 100

metricas = {
    "Publicaciones únicas": len(df_productos),
    "Completitud de campos (%)": round(completitud, 2),
    "URLs duplicadas": int(df_productos["URL"].duplicated().sum()),
    "Precios no numéricos": int(df_productos[["Precio mínimo", "Precio máximo"]]
                                .isna().sum().sum()),
    "Respuestas HTTP distintas de 200": len(errores_http),
}

for nombre, valor in metricas.items():
    print(f"{nombre}: {valor}")

display(df_productos)


Publicaciones únicas: 26
Completitud de campos (%): 100.0
URLs duplicadas: 0
Precios no numéricos: 0
Respuestas HTTP distintas de 200: 0


,Nombre,Marca,SKU de referencia,Precio mínimo,Precio máximo,Moneda,URL
0,Tenis Deportivo de Mujer Skechers 117736wht,Skechers,461817,279900,279900,COP,https://www.calzatodo.com.co/tenis-117736wht-2...
1,Tenis Deportivo de Mujer Skechers 117759ntol,Skechers,461823,249900,249900,COP,https://www.calzatodo.com.co/tenis-117759ntol-...
2,Tenis Deportivo de Mujer Skechers 117759wbl,Skechers,461829,249900,249900,COP,https://www.calzatodo.com.co/tenis-117759wbl-2...
3,Tenis Deportivo de Mujer Skechers 117736blk,Skechers,461811,279900,279900,COP,https://www.calzatodo.com.co/tenis-117736blk-2...
4,Tenis Deportivo de Mujer Throwing .aa5133w,Throwing,451564,139900,139900,COP,https://www.calzatodo.com.co/tenis-aa5133w-712...
5,Tenis Deportivo de Mujer Throwing .ae6060w,Throwing,451589,139900,139900,COP,https://www.calzatodo.com.co/tenis-ae6060w-712...
6,Tenis Deportivo de Mujer Throwing .au52830w eco,Throwing,451576,139900,139900,COP,https://www.calzatodo.com.co/tenis-au52830w-ec...
7,Tenis Deportivo de Mujer Throwing . au52199w,Throwing,450288,139900,139900,COP,https://www.calzatodo.com.co/tenis--au52199w-7...
8,Tenis Deportivo de Mujer Throwing . aw54154w,Throwing,450312,139900,139900,COP,https://www.calzatodo.com.co/tenis--aw54154w-7...
9,Tenis Deportivo de Mujer Throwing . af56591weco,Throwing,450301,139900,139900,COP,https://www.calzatodo.com.co/tenis--af56591wec...


## 4. Almacenamiento

La tabla se exporta a CSV separado por punto y coma, con codificación UTF-8 con
BOM para que Excel reconozca las tildes.


In [4]:
df_productos.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8-sig", sep=";")

ruta = os.path.abspath(ARCHIVO_CSV)
print("Archivo generado:", ruta)
print("Registros escritos:", len(df_productos))
print("Precio mínimo del catálogo:", df_productos["Precio mínimo"].min())
print("Precio máximo del catálogo:", df_productos["Precio máximo"].max())

if EN_COLAB:
    from google.colab import files

    files.download(ARCHIVO_CSV)


Archivo generado: /home/andresosoa/Documentos/Universidad/Analitica_Datos/EA2_revision/entrega/calzatodo_tenis_deportivos_mujer.csv
Registros escritos: 26
Precio mínimo del catálogo: 134900
Precio máximo del catálogo: 349900


## 5. Control de versiones con Git y GitHub

El flujo que se documenta a continuación es el que se siguió en la sesión de
trabajo: clonar el repositorio, incorporar los resultados, registrarlos con un
commit descriptivo y publicarlos en la rama `main`.

La clonación es idempotente: si la carpeta ya existe se reutiliza en lugar de
volver a descargarla, de modo que el notebook puede ejecutarse varias veces sin
errores.

El token de acceso se lee de la variable de entorno `GITHUB_TOKEN` o de los
Secretos de Colab. Nunca se escribe en el notebook ni en el repositorio.


In [5]:
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repositorio clonado en", REPO_DIR)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    print("Repositorio ya presente en", REPO_DIR, "(se actualizó si era posible)")

subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-5"], check=False)


Clonando en 'EA2_Web_Scraping'...


Repositorio clonado en EA2_Web_Scraping
a228133 Merge pull request #1 from Estefa-M/mejora-readme
bad53ad Update README.md
21e98cb Update README.md
debc2f6 Creado con Colab
8cef96e Agrega CSV de tenis deportivos de Calzatodo


CompletedProcess(args=['git', '-C', 'EA2_Web_Scraping', 'log', '--oneline', '-5'], returncode=0)

In [6]:
import shutil

shutil.copy(ARCHIVO_CSV, os.path.join(REPO_DIR, ARCHIVO_CSV))

subprocess.run(["git", "-C", REPO_DIR, "add", ARCHIVO_CSV], check=True)
subprocess.run(["git", "-C", REPO_DIR, "status", "--short"], check=True)

print("\nArchivos versionados en el repositorio:")
for nombre in sorted(os.listdir(REPO_DIR)):
    if not nombre.startswith("."):
        print(" -", nombre)


M  calzatodo_tenis_deportivos_mujer.csv

Archivos versionados en el repositorio:
 - README.md
 - calzatodo_ea1.ipynb
 - calzatodo_tenis_deportivos_mujer.csv


In [7]:
subprocess.run(
    ["git", "-C", REPO_DIR, "commit",
     "-m", "Actualiza la extraccion de Calzatodo con la paginacion corregida"],
    check=False,
)


[main 4e7d1f4] Actualiza la extraccion de Calzatodo con la paginacion corregida
 1 file changed, 8 insertions(+), 6 deletions(-)


CompletedProcess(args=['git', '-C', 'EA2_Web_Scraping', 'commit', '-m', 'Actualiza la extraccion de Calzatodo con la paginacion corregida'], returncode=0)

In [8]:
token = os.environ.get("GITHUB_TOKEN")

if EN_COLAB and not token:
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None

if token:
    entorno = {**os.environ, "GITHUB_TOKEN": token}
    resultado = subprocess.run(
        ["git", "-C", REPO_DIR,
         "-c", "credential.helper=",
         "-c", 'credential.helper=!f() { printf "username=Estefa-M\\npassword=%s\\n" "$GITHUB_TOKEN"; }; f',
         "push", "origin", "main"],
        env=entorno, capture_output=True, text=True,
    )
    print(resultado.stdout or resultado.stderr)
    print("Código de salida del push:", resultado.returncode)
else:
    print("GITHUB_TOKEN no está definido en este entorno.")
    print("Los cambios quedaron registrados localmente; para publicarlos, defina")
    print("el token y vuelva a ejecutar esta celda.")


GITHUB_TOKEN no está definido en este entorno.
Los cambios quedaron registrados localmente; para publicarlos, defina
el token y vuelva a ejecutar esta celda.


## 6. Cómo reproducir la ejecución

1. Abrir el notebook en Google Colab.
2. Ejecutar las celdas en orden (`Entorno de ejecución > Ejecutar todas`).
3. Para publicar en GitHub, crear un token de acceso personal con permiso sobre
   el repositorio y guardarlo como secreto `GITHUB_TOKEN` en Colab.
4. El archivo `calzatodo_tenis_deportivos_mujer.csv` se genera en la carpeta de
   trabajo y se descarga automáticamente cuando se ejecuta en Colab.
